#**2nd Week**

##**Задачи - Подзапросы (Оба уровня)**

В этом тесте вам предстоит решить практические задачи на тему "Подзапросы".

Найти все бронирования, совершенные клиентами с именем "Иван". Выведите всю информацию по данным бронированиям.

In [ ]:
SELECT *
FROM bookings
WHERE renter_id IN (
    SELECT id
    FROM clients
    WHERE first_name = 'Иван'
);

Выведите список номеров (room_number), которые не были забронированы ни разу.

In [ ]:
SELECT room_number
FROM rooms
WHERE room_number NOT IN (
    SELECT room_number
    FROM bookings
);

Выведите всю информацию о первых 50 бронированиях, для которых была совершена оплата в период с 1 января 2020 года по 31 марта 2020 года, отсортировав их по id бронирования.

In [ ]:
SELECT b.*
FROM bookings b
JOIN payments p ON b.booking_id = p.booking_id
WHERE p.payment_date BETWEEN '2020-01-01' AND '2020-03-31'
ORDER BY b.booking_id
LIMIT 50;

Выведите первые 50 клиентов (id, имя и фамилию), которые не совершали бронирования, отсортировав их по возрастанию id.

In [ ]:
SELECT c.id, c.first_name, c.last_name
FROM clients c
LEFT JOIN bookings b ON c.id = b.renter_id
WHERE b.booking_id IS NULL
ORDER BY c.id
LIMIT 50;

Выведите первые 50 клиентов (id, имя и фамилию), которые забронировали номер, стоимость которого больше 20000. Упорядочьте их по id.

In [ ]:
SELECT DISTINCT c.id, c.first_name, c.last_name
FROM clients c
JOIN bookings b ON c.id = b.renter_id
JOIN rooms r ON b.room_number = r.room_number
WHERE r.price_per_night > 20000
ORDER BY c.id
LIMIT 50;

Выведите первые 50 идентификаторов клиентов, которые бронировали номер, с ценой за ночь выше средней стоимости всех номеров, отсортировав их по возрастанию. Идентификаторы должны быть уникальными.

In [ ]:
SELECT DISTINCT b.renter_id
FROM bookings b
JOIN rooms r ON b.room_number = r.room_number
WHERE r.price_per_night > (SELECT AVG(price_per_night) FROM rooms)
ORDER BY b.renter_id
LIMIT 50;

Определить количество клиентов, которые забронировали номера категории 'Люкс'. Назовите единственную колонку полученной таблицы lux_cnt.

In [ ]:
SELECT COUNT(DISTINCT b.renter_id) AS lux_cnt
FROM bookings b
JOIN rooms r ON b.room_number = r.room_number
WHERE r.type_name = 'Люкс';

Выведите всю информацию о первых 50 бронированиях, срок которых меньше, чем средний срок бронирования. Отсортируйте их по id бронирования.

In [ ]:
SELECT *
FROM bookings
ORDER BY booking_id
LIMIT 50;

Выведите первые 50 идентификаторов клиентов, которые сделали бронирование только на номера с максимальной возможной вместимостью. Отсортируйте их по id клиента. Идентификаторы клиентов должны быть уникальными.

In [ ]:
WITH max_occupancy AS (
    SELECT MAX(max_occupancy) AS max_occupancy
    FROM rooms
),
client_bookings AS (
    SELECT b.renter_id, r.max_occupancy
    FROM bookings b
    JOIN rooms r ON b.room_number = r.room_number
),
filtered_clients AS (
    SELECT renter_id
    FROM client_bookings
    GROUP BY renter_id
    HAVING 
        -- Клиент бронировал только один тип номеров по вместимости
        COUNT(DISTINCT max_occupancy) = 1 
        -- И этот тип имеет максимальную вместимость
        AND MAX(max_occupancy) = (SELECT max_occupancy FROM max_occupancy)
)
SELECT renter_id
FROM filtered_clients
ORDER BY renter_id
LIMIT 50;